In [1]:
#Currently supports 1 race at a time, will implement vectorized environments later
import pystk2
import random

In [2]:
#Currently using HD graphics for debugging. Will disable graphics later
GraphicsConfig = pystk2.GraphicsConfig.hd()
pystk2.init(GraphicsConfig)

libdecor-gtk-WARNING: Failed to initialize GTK
Failed to load plugin 'libdecor-gtk.so': failed to init
No plugins found, falling back on no decorations

(python:33732): Gtk-WARNING **: 18:28:40.327: gtk_disable_setlocale() must be called before gtk_init()
libdecor-gtk-WARNING: Failed to initialize GTK
Failed to load plugin 'libdecor-gtk.so': failed to init
No plugins found, falling back on no decorations


..:: Antarctica Rendering Engine 2.0 ::..


In [3]:
WorldState = pystk2.WorldState()

In [4]:
RaceConfig = pystk2.RaceConfig(
	track='lighthouse',
	num_kart=1,
	laps=1
)

RaceConfig.players[0].controller = pystk2.PlayerConfig.Controller.PLAYER_CONTROL

race = pystk2.Race(RaceConfig)

In [5]:
from collections import deque
import numpy as np

class ProcessState:
    def __init__(self, max_speed=30, map_size=100, track_length=2000):
        self.frame = deque(maxlen=4)    # Window holding last 4 frames
        self.max_speed = max_speed
        self.map_size = map_size
        self.track_length = track_length

    def processObservation(self, obs):
        loc = np.array(obs["location"], dtype=np.float32) / self.map_size
        loc = np.clip(loc, -1.0, 1.0)

        vel = np.array(obs["velocity"], dtype=np.float32) / self.max_speed
        vel = np.clip(vel, -1.0, 1.0)

        front = np.array(obs["front"], dtype=np.float32) / self.map_size
        front = np.clip(front, -1.0, 1.0)

        jump = np.array([1.0 if obs["jumping"] else 0.0], dtype=np.float32)

        rotation = np.array(obs["rotation"], dtype=np.float32)

        dist = np.array([obs.get("distance_down_track", 0.0)], dtype=np.float32) / self.track_length

        state = np.concatenate([loc, vel, front, jump, rotation, dist])

        if len(self.frame) == 0:
            for _ in range(4):
                self.frame.append(state)
        else:
            self.frame.append(state)

        return np.concatenate(self.frame)
    



In [6]:
import numpy as np
race.start()
#Track details
track = pystk2.Track()
track.update()
track_length = track.length
max_coordinate = np.max(np.abs(track.path_nodes))
#actions are currently random, make sure that the outputs are in the range specified.
action = pystk2.Action(
	acceleration = random.random(),
	brake = True if random.randint(0,1) == 1 else False,
	drift = True if random.randint(0,1) == 1 else False,
	fire = True if random.randint(0,1) == 1 else False,
	nitro = True if random.randint(0,1) == 1 else False,
	#rescue = True if random.randint(0,1) == 1 else False,
	steer = random.uniform(-1,1)
)

processor = ProcessState(max_speed=30, map_size=max_coordinate, track_length=track_length)
buffer = []

for _ in range(1000):
	RaceEnded = race.step(action)
	WorldState.update()
	#race.render_data has the stuff you need to translate
	for data in race.render_data:
		color = data.image
		depth = data.depth
		labels = data.instance
	#https://pystk.readthedocs.io/en/latest/state.html Check this link for more info on states
	kart = WorldState.karts[0]
	kart_state = []
	obs = {
		"location": kart.location,
		"velocity": kart.velocity,
		"front": kart.front,
		"jumping": kart.jumping,
		"rotation": kart.rotation,
		"distance_down_track": kart.distance_down_track
	}
	kart_state.append(obs)
	print(f'Distance: {WorldState.karts[0].distance_down_track} | FinishTime: {WorldState.karts[0].finish_time} | FinishedLaps: {WorldState.karts[0].finished_laps} | Front: {WorldState.karts[0].front} | ID: {WorldState.karts[0].id} | Jumping: {WorldState.karts[0].jumping} | LapTime: {WorldState.karts[0].lap_time} | Location: {WorldState.karts[0].location} | MaxSteerAngle: {WorldState.karts[0].max_steer_angle} | Name: {WorldState.karts[0].name} | OverallDistance: {WorldState.karts[0].overall_distance} | PlayerID: {WorldState.karts[0].player_id} | PowerUp: {WorldState.karts[0].powerup} | RaceResult: {WorldState.karts[0].race_result} | Rotation: {WorldState.karts[0].rotation} | ShieldTime: {WorldState.karts[0].shield_time} | Size: {WorldState.karts[0].size} | Velocity: {WorldState.karts[0].velocity} | WheelBase: {WorldState.karts[0].wheel_base}')
	np_obs = processor.processObservation(obs=obs)
	print(np_obs)
	
	# Reward has to added.
	transition = {
		"state": np_obs,
		"action": np.array([action.acceleration, action.steer, float(action.brake), float(action.drift), float(action.nitro)], dtype=np.float32),
		"done": RaceEnded
	}
	buffer.append(transition)

Distance: 0.0 | FinishTime: 0.0 | FinishedLaps: -1 | Front: [ 17.527885 -13.898107 -50.688072] | ID: 0 | Jumping: False | LapTime: 17895698.0 | Location: [ 16.810137 -13.866424 -50.698967] | MaxSteerAngle: 0.4247603416442871 | Name: Tux | OverallDistance: -879.8558959960938 | PlayerID: 0 | PowerUp: <Powerup type=<Type.NOTHING: 0> num=0> | RaceResult: False | Rotation: [ 0.7122851  -0.0153278  -0.7015585   0.01519059] | ShieldTime: 0.0 | Size: [0.821 0.675 1.437] | Velocity: [0.10875531 0.04264199 0.00203452] | WheelBase: 1.0084210634231567
[ 9.5010363e-02 -7.8372590e-02 -2.8654897e-01  3.6251768e-03
  1.4213996e-03  6.7817280e-05  9.9067055e-02 -7.8551665e-02
 -2.8648740e-01  0.0000000e+00  7.1228510e-01 -1.5327798e-02
 -7.0155847e-01  1.5190593e-02  0.0000000e+00  9.5010363e-02
 -7.8372590e-02 -2.8654897e-01  3.6251768e-03  1.4213996e-03
  6.7817280e-05  9.9067055e-02 -7.8551665e-02 -2.8648740e-01
  0.0000000e+00  7.1228510e-01 -1.5327798e-02 -7.0155847e-01
  1.5190593e-02  0.0000000e

In [7]:
race.stop()
del race
pystk2.clean()

  wl_callback#49 still attached
  wl_surface#40 still attached
